In [41]:
import pandas as pd
from pyecharts.charts import Map
from pyecharts import options as opts
from pyecharts.charts import Line
from datetime import datetime
from pyecharts.charts import Page
from pyecharts.charts import Pie
import re
from pyecharts.charts import Bar

In [44]:
df = pd.read_csv('Alldata.csv', encoding='utf-8')
exclude_provinces = ["海外"]
# 数据清洗
# 删除地址为空或NaN的数据
df = df.dropna(subset=['地址'])
df = df[df['地址'] != '']
province_mask = df['地址'].apply(lambda x: any(province in str(x) for province in exclude_provinces))
df = df[~province_mask].copy()

# 统计各省份企业数量
province_counts = df['地址'].value_counts().reset_index()
province_counts.to_csv('province_counts.csv', encoding='utf-8',index=False)
province_counts.columns = ['省份', '企业数量']

# 准备热力图数据（格式：[(省份, 数量), ...]）
heatmap_data = [(prov, count) for prov, count in zip(province_counts['省份'], province_counts['企业数量'])]

province_mapping = {
    '北京': '北京市',
    '广东': '广东省',
    '上海': '上海市',
    '浙江': '浙江省',
    '江苏': '江苏省',
    '山东': '山东省',
    '福建': '福建省',
    '四川': '四川省',
    '湖北': '湖北省',
    '河南': '河南省',
    '湖南': '湖南省',
    '安徽': '安徽省',
    '辽宁': '辽宁省',
    '重庆': '重庆市',
    '河北': '河北省',
    '天津': '天津市',
    '陕西': '陕西省',
    '江西': '江西省',
    '广西': '广西壮族自治区',
    '云南': '云南省',
    '黑龙江': '黑龙江省',
    '海南': '海南省',
    '山西': '山西省',
    '吉林': '吉林省',
    '贵州': '贵州省',
    '新疆': '新疆维吾尔自治区',
    '内蒙古': '内蒙古自治区',
    '甘肃': '甘肃省',
    '宁夏': '宁夏回族自治区',
    '西藏': '西藏自治区',
    '青海': '青海省',
    '香港': '香港特别行政区'
}
# 转换为全称
converted_data = [(province_mapping[prov], count) for prov, count in heatmap_data]
# print(converted_data)

# 创建地图实例
china_map = Map(init_opts=opts.InitOpts(width="1300px", height="1000px"))

# 添加数据（maptype="china"）
china_map.add(
    series_name="企业数量",
    data_pair=converted_data,
    maptype="china",
    is_map_symbol_show=False,
    label_opts=opts.LabelOpts(is_show=True)  # 显示省份标签
)

# 设置全局配置
china_map.set_global_opts(
    title_opts=opts.TitleOpts(title="中国企业地域分布", 
                              pos_left="center",
                              pos_top="1%",       # 标题上移
                              padding=[0, 0, 30, 0]  # 增加下边距
                             ),
    visualmap_opts=opts.VisualMapOpts(
        min_=0,
        max_=2600,
        range_text=["高", "低"],
        is_piecewise=True,
        pos_left="center",
        pos_bottom="5%",
        textstyle_opts=opts.TextStyleOpts(color="#000"),
    ),
    legend_opts=opts.LegendOpts(pos_top="8%"),  # 图例下移
)

# 读取CSV文件
df = pd.read_csv('Alldata.csv', encoding='utf-8')

# 数据清洗
def parse_registration_date(date_str):
    try:
        return pd.to_datetime(date_str)
    except:
        return None

df['注册时间'] = df['注册时间'].apply(parse_registration_date)
df = df.dropna(subset=['注册时间'])

# 筛选2000-2010年的数据
df = df[(df['注册时间'] >= '2000-01-01') & (df['注册时间'] <= '2010-12-31')]

# 按年份统计注册数量
df['注册年份'] = df['注册时间'].dt.year
yearly_counts = df['注册年份'].value_counts().sort_index()

# 补全缺失年份
all_years = range(2000, 2011)
yearly_counts = yearly_counts.reindex(all_years, fill_value=0)

# print("2000-2010年企业注册数量统计：")
# print(yearly_counts)
yearly_counts.to_csv('yearly_counts.csv', encoding='utf-8',index=False)
# 绘制折线图
line = (
    Line(init_opts=opts.InitOpts(width="1000px", height="500px"))
    .add_xaxis(xaxis_data=[str(y) for y in all_years])  # 确保x轴为字符串类型
    .add_yaxis(
        series_name="企业注册数量",
        y_axis=yearly_counts.tolist(),
        is_smooth=True,
        symbol="circle",
        symbol_size=8,
        label_opts=opts.LabelOpts(is_show=True),
        linestyle_opts=opts.LineStyleOpts(width=3),
        itemstyle_opts=opts.ItemStyleOpts(
            border_width=2,
            border_color="#fff",
            color="#c23531"
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="2000-2010年企业注册数量变化趋势",
            subtitle=f"总企业数: {yearly_counts.sum()}家",
            pos_left="center"
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="axis",
            formatter="{a}<br/>{b}: {c}家"
        ),
        xaxis_opts=opts.AxisOpts(
            name="年份",
            type_="category",  # 明确指定为类别轴
            boundary_gap=False
        ),
        yaxis_opts=opts.AxisOpts(
            name="注册数量",
            type_="value",
            splitline_opts=opts.SplitLineOpts(is_show=True)
        ),
        legend_opts=opts.LegendOpts(
            pos_right="4%",
            pos_top="15%",
            orient="vertical",
            item_width=50,
            item_height=20,
            textstyle_opts=opts.TextStyleOpts(font_size=12)
        ),
    )
)

# 读取CSV文件
df = pd.read_csv('Alldata.csv')

# 处理数据：删除企业类型为空的行
df = df.dropna(subset=['类型'])
df = df[
    ~df['类型'].isin(['', '--', '110000005118165', '未公开']) &  # 排除空字符串和'--'
    df['类型'].notna()  # 排除NaN值
]
# 清洗企业类型字段：去除括号及括号内容
def clean_company_type(type_str):
    # 使用正则表达式匹配并去除括号及括号内容
    cleaned_type = re.sub(r'[（(].*?[)）]', '', type_str)
    return cleaned_type.strip()

df['清洗后类型'] = df['类型'].apply(clean_company_type)

# 统计各类型数量
type_counts = df['清洗后类型'].value_counts()
type_counts.to_csv('type_counts.csv', encoding='utf-8',index=False)
# 准备环形图数据
data = [(name, count) for name, count in type_counts.items()]

# 创建环形图
pie1 = (
    Pie(init_opts=opts.InitOpts(width="2400px", height="1200px"))
    .add(
        "",
        data,
        radius=["25%", "50%"],  # 内半径和外半径，形成环形效果
        center=["50%", "55%"],  # 整体下移5%
        label_opts=opts.LabelOpts(
            position="outside",
            formatter="{b}: {c} ({d}%)",  # 显示名称、数量和百分比
        ),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="企业类型占比环形图", 
                                  pos_left="center",    # 标题基准定位在右侧
                                  pos_top="1%", 
                                 ),
        legend_opts=opts.LegendOpts(
            orient="vertical",  # 垂直排列
            pos_right="15%",     # 距离右侧5%的位置
            pos_top="center",   # 垂直居中
            item_width=20,      # 图例标记的图形宽度
            item_height=10,     # 图例标记的图形高度
            item_gap=5          # 图例每项之间的间隔
        ),
    )
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(
            trigger="item", formatter="{a} <br/>{b}: {c} ({d}%)"
        ),
    )
)


## 读取CSV文件
df = pd.read_csv('Alldata.csv')

## 数据清洗
# 删除行业为空值的数据
df = df.dropna(subset=['行业'])

# 由于数据错误，行业列里面是省份名称，需要排除的省份列表
exclude_provinces = ["北京", "广东", "山东", "四川", "安徽", "江苏", "辽宁", "宁夏", 
                    "浙江", "湖北", "陕西", "重庆", "内蒙古", "天津", "云南", "上海"]

# 删除行业列中包含在排除列表中的数据
df = df[~df['行业'].isin(exclude_provinces)]
industry_counts.to_csv('industry_counts.csv', encoding='utf-8',index=False)
# 按行业分组计算企业数量
industry_counts = df.groupby('行业').size().reset_index(name='数量')
# 排序或者乱序
# industry_counts = industry_counts.sample(frac=1).reset_index(drop=True)
industry_counts = industry_counts.sort_values(by='数量').reset_index(drop=True)
# print(industry_counts)
# 准备数据
data_pair = list(industry_counts[['行业', '数量']].itertuples(index=False, name=None))

# 绘制环形图
pie2 = Pie(init_opts=opts.InitOpts(width="1200px", height="600px"))

pie2.add(
    series_name="各行业企业数量占比",
    data_pair=data_pair,  # 指定数据
    radius=["40%", "75%"],  # 内半径40%，外半径75%，形成环形
    label_opts=opts.LabelOpts(
        formatter="{b}: {d}%",                     
    ),  # 显示百分比数据
    # rosetype="radius"  # 若需要玫瑰图效果可取消注释
)


pie2.set_global_opts(
    title_opts=opts.TitleOpts(title="各行业企业数量占比环形图", pos_left="center",),
    legend_opts=opts.LegendOpts(orient="vertical", 
        pos_right="1%",       # 距离右侧5%的位置
        pos_top="bottom",     # 垂直居中
        item_width=14,        # 图例标记宽度
        item_height=14,        # 图例标记高度               
    )
)


# 由于数据错误，行业列里面是省份名称，需要排除的省份列表
exclude_provinces = ["北京", "广东", "山东", "四川", "安徽", "江苏", "辽宁", "宁夏", 
                    "浙江", "湖北", "陕西", "重庆", "内蒙古", "天津", "云南", "上海"]

# 读取数据
df = pd.read_csv('Alldata.csv', encoding='utf-8')

# 数据清洗
# 删除关键字段为空的数据
df = df.dropna(subset=['市值', '行业'])

# 删除行业列中包含省份名称的数据行
province_mask = df['行业'].apply(lambda x: any(province in str(x) for province in exclude_provinces))
# print(f"\n发现市值列包含省份的记录数: {province_mask.sum()}")
# print("将被删除的示例数据:")
# print(df[province_mask].head()[['公司全称', '市值']])

df = df[~province_mask].copy()
# print(f"删除后剩余数据量: {len(df)}")

# 市值单位转换函数
def convert_market_value(value):
    try: 
        # 提取数字部分
        num_str = ''.join([c for c in str(value).split('：')[1] if c.isdigit() or c == '.'])
        num = float(num_str)
        
        if '亿' in str(value):
            return num
        elif '万' in str(value):
            return num / 10000
        elif '美元' in str(value):
            return num * 7 / 10000 if '万' in str(value) else num * 7
        return num  # 默认按亿处理
    except:
        # print(f"无法解析的市值值: {value}")
        return None

df['市值_亿'] = df['市值'].apply(convert_market_value)
df = df.dropna(subset=['市值_亿'])
df.to_csv('convert_market_value.csv', encoding='utf-8',index=False)
# 按行业统计，使用清洗后的行业列
industry_data = (
    df.groupby('行业', as_index=False)['市值_亿']
    .sum()
    .sort_values('市值_亿', ascending=False)
)

# 绘制饼图
data_pair = [(row['行业'], round(row['市值_亿'], 2)) for _, row in industry_data.iterrows()]

pie3 = (
    Pie(init_opts=opts.InitOpts(width="1000px", height="600px"))
    .add(
        series_name="市值(亿元)",
        data_pair=data_pair,
        radius=["0%", "65%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(
            formatter="{b}|{c}亿|{d}%",
            font_size=12,
            rich={
                "b": {"fontSize": 14, "lineHeight": 33},
                "per": {"color": "#eee", "backgroundColor": "#334455"}
            }
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="各行业市值占比分析",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=20)
        ),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_right="1%",
            pos_top="15%",
            item_width=40,
            item_height=20
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter="{a}<br/>{b}: {c}亿元 ({d}%)"
        )
    )
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        label_opts=opts.LabelOpts(
            position="outside",
            formatter="{b}: {d}%",
        )
    )
)


# 读取CSV文件
df = pd.read_csv('Alldata.csv', encoding='utf-8')

# 数据清洗
# 处理证券类别字段
df['证券类别'] = df['证券类别'].str.strip()  # 去除前后空格
# 删除无效数据（空值或'--'）
df = df[
    ~df['证券类别'].isin(['', '--']) &  # 排除空字符串和'--'
    df['证券类别'].notna()  # 排除NaN值
]

# print(df[:5])
# 统计各证券类别的数量
category_counts = df['证券类别'].value_counts().reset_index()
category_counts.to_csv('category_counts.csv', encoding='utf-8',index=False)
category_counts.columns = ['证券类别', '数量']

# 绘制柱形图
bar = (
    Bar(init_opts=opts.InitOpts(width="1200px", height="600px"))
    .add_xaxis(category_counts['证券类别'].tolist())
    .add_yaxis(
        "企业数量",
        category_counts['数量'].tolist(),
        category_gap="60%",  # 柱子间距
        itemstyle_opts=opts.ItemStyleOpts(color="#5470C6")  # 柱子颜色
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="企业证券类别数量分布",
            subtitle=f"有效数据量: {len(df)}条",
            pos_left="center"
        ),
        xaxis_opts=opts.AxisOpts(
            name="证券类别",
            axislabel_opts=opts.LabelOpts(rotate=45)  # x轴标签旋转45度防重叠
        ),
        yaxis_opts=opts.AxisOpts(
            name="企业数量",
            splitline_opts=opts.SplitLineOpts(is_show=True)  # 显示横向网格线
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="axis",
            formatter="{b}<br/>企业数量: {c}"
        ),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_right="5%",
            pos_top="15%",
            item_width=40,
            item_height=20
        ),
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(
            position="top",
            formatter="{c}"  # 在柱子上方显示数值
        )
    )
)

page = Page(layout=Page.SimplePageLayout)  # 简单布局，图表垂直排列
page.add(pie3, china_map, line, pie1, bar, pie2)

# 渲染所有图表到一个HTML文件
page.render("multiple_charts.html")

'E:\\pythonProject\\数据可视化\\企业数据可视化\\multiple_charts.html'